<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/10-Adding_Reranking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Improving Retrieval with Cohere Reranking

This notebook demonstrates how to add **Cohere reranking** as a post-processing step on top of LlamaIndex vector retrieval to improve the quality of retrieved context for RAG pipelines.

## What You'll Learn
- How to load a pre-built vector store from HuggingFace Hub
- How to configure `CohereRerank` as a node post-processor in LlamaIndex
- How to run a query with reranking and inspect the reranked results
- How to evaluate retrieval quality (hit rate and MRR) with and without reranking using `RetrieverEvaluator`

## Prerequisites
- Familiarity with RAG pipelines and vector stores (see notebooks 02–09)
- A working OpenAI API key (for embeddings and LLM)
- A working Cohere API key (for the reranker)
- Basic understanding of LlamaIndex query engines and retrievers

## Install Packages and Setup Variables

In [ ]:
!pip install -q llama-index==0.14.0 openai==1.107.0 llama-index-finetuning==0.4.1 llama-index-postprocessor-cohere-rerank==0.5.1 \
                llama-index-embeddings-huggingface==0.6.1 llama-index-embeddings-cohere==0.6.1 llama-index-embeddings-openai==0.5.2 cohere==5.18.0 \
                chromadb==1.0.21 llama-index-vector-stores-chroma==0.5.3 llama-index-llms-google-genai==0.5.0 jedi==0.19.2

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os

# Set the following API keys in the Python environment. They will be used later.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
# os.environ["CO_API_KEY"] = "YOUR_COHERE_API_KEY"
# os.environ["COHERE_API_KEY"] = "YOUR_COHERE_API_KEY"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
    os.environ["CO_API_KEY"] = userdata.get('COHERE_API_KEY')
    os.environ["COHERE_API_KEY"] = userdata.get('COHERE_API_KEY')
    cohere_key = userdata.get('COHERE_API_KEY')
except ImportError:
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "")
    os.environ["GOOGLE_API_KEY"] = os.environ.get("GOOGLE_API_KEY", "")
    os.environ["CO_API_KEY"] = os.environ.get("COHERE_API_KEY", "")
    cohere_key = os.environ.get("COHERE_API_KEY", "")

# Initialize Embedding and LLM Models

In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-5-mini", additional_kwargs={'reasoning_effort':'minimal'})
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

### Download vector store

In [ ]:
# Run this after downloading the vector store from HuggingFace Hub above
!unzip -o vectorstore.zip

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Create the vector index
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
index = VectorStoreIndex.from_vector_store(vector_store)

# Query Dataset

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank

# Define the Cohere Reranking object to return only the two highest-ranking chunks.
cohere_rerank3 = CohereRerank(top_n=2, api_key=cohere_key, model='rerank-english-v3.0')

In [ ]:
# Define the query engine with the LLM for generating the final answer
# and the embedding model for retrieving related nodes.
# The `node_postprocessors` step will be applied to the retrieved nodes.
query_engine = index.as_query_engine(
    similarity_top_k=5, node_postprocessors=[cohere_rerank3]
)

res = query_engine.query("Write about Retrieval Augmented Generation?")

In [ ]:
print(res.response)

In [ ]:
# Show the retrieved nodes
for src in res.source_nodes:
    print("Node ID\t", src.node_id)
    print("Title\t", src.metadata["title"])
    print("Text\t", src.text)
    print("Score\t", src.score)
    print("-_" * 20)

## Optional: Reranking Score Comparison

In [ ]:
# Compare relevance scores before and after Cohere reranking
print("=== Node Scores After Cohere Reranking ===\n")
print(f"{'Rank':<6} {'Score':>8}  {'Text preview'}")
print("-" * 70)
for rank, node in enumerate(res.source_nodes, 1):
    score = f"{node.score:.4f}" if node.score is not None else "  N/A "
    preview = node.text[:80].replace("\n", " ")
    print(f"{rank:<6} {score:>8}  {preview}...")

print(f"\nTotal nodes returned after reranking: {len(res.source_nodes)}")
print("Note: Cohere reranker re-scores all retrieved nodes and returns the top-N most relevant.")

# Evaluate Cohere rerank

In [ ]:
from huggingface_hub import hf_hub_download
from llama_index.core.evaluation import EmbeddingQAFinetuneDataset

# Download the evaluation dataset
hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="rag_eval_dataset_question_context_subset_50.json", repo_type="dataset", local_dir="./")
rag_eval_dataset = EmbeddingQAFinetuneDataset.from_json("./rag_eval_dataset_question_context_subset_50.json")

In [ ]:
import asyncio
import pandas as pd
from llama_index.core.evaluation import RetrieverEvaluator


def display_results_retriever(name, eval_results):
    """Format retriever evaluation results as a simple summary string."""
    metric_dicts = [r.metric_vals_dict for r in eval_results]
    full_df = pd.DataFrame(metric_dicts)
    hit_rate = full_df["hit_rate"].mean()
    mrr = full_df["mrr"].mean()
    return f"{name}: Hit Rate = {hit_rate:.4f}, MRR = {mrr:.4f}"


# We can evaluate the retrievers with different top_k values.
loop = asyncio.get_event_loop()
for i in [2, 4, 6, 8, 10]:
    retriever = index.as_retriever(
        similarity_top_k=i, node_postprocessors=[cohere_rerank3]
    )
    retriever_evaluator = RetrieverEvaluator.from_metric_names(
        ["mrr", "hit_rate"], retriever=retriever
    )
    eval_results = loop.run_until_complete(
        retriever_evaluator.aevaluate_dataset(rag_eval_dataset)
    )
    print(display_results_retriever(f"Retriever top_{i}", eval_results))

## Optional: With vs Without Reranking

In [ ]:
# Compare the response quality with and without the Cohere reranker
test_query = "What are the key differences between LLaMA 1 and LLaMA 2?"

print(f"Query: {test_query}\n")
print("=" * 65)

# Without reranking
try:
    plain_engine = index.as_query_engine(similarity_top_k=5)
    plain_res = plain_engine.query(test_query)
    print("WITHOUT reranking:")
    print(str(plain_res)[:500])
    print(f"(Retrieved {len(plain_res.source_nodes)} nodes)")
except Exception as e:
    print(f"Without reranking error: {e}")

print("\n" + "-" * 65)

# With reranking
try:
    rerank_engine = index.as_query_engine(
        similarity_top_k=10,
        node_postprocessors=[CohereRerank(api_key=os.environ.get("COHERE_API_KEY", ""), top_n=3)],
    )
    rerank_res = rerank_engine.query(test_query)
    print("WITH Cohere reranking (top_n=3 from 10):")
    print(str(rerank_res)[:500])
    print(f"(Retrieved {len(rerank_res.source_nodes)} nodes after reranking)")
except Exception as e:
    print(f"With reranking error: {e}")